In [9]:
import os
import re
import csv
import json
from pathlib import Path
from rapidfuzz import process

# =========================================================
# 1. ใส่ Error Log ที่ได้จากระบบตรวจสอบไว้ตรงนี้
# =========================================================
error_log = """
🔍 เริ่มตรวจสอบความครบถ้วนและความถูกต้องของรายชื่อ...

📌 [บช] อำเภอ: งาว | ตำบล: นาแก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านร้อง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านอ้อน | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านอ้อน | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านอ้อน | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 55 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 2 รายชื่อ: เพื่อชาติไทย, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านอ้อน | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านแหง | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านโป่ง | หน่วยที่: 10
   📊 มีข้อมูลทั้งหมด 23 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 34 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ทางเลือกใหม่, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ประชาธิปไตยใหม่, ปวงชนไทย, พลวัต, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, มิติใหม่, รวมพลังประชาชน, รวมใจไทย, รวมไทยสร้างชาติ, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชาติไทย, เพื่อชีวิตใหม่, เพื่อไทย, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ใหม่, ไทยก้าวหน้า, ไทยชนะ, ไทยทรัพย์ทวี, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: บ้านโป่ง | หน่วยที่: 11
   📊 มีข้อมูลทั้งหมด 10 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 47 รายชื่อ: กรีน, กล้าธรรม, ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ความหวังใหม่, ท้องที่ไทย, ประชากรไทย, ประชาชน, ประชาชาติ, ประชาธิปัตย์, ประชาอาสาชาติ, ประชาไทย, ปวงชนไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังสังคมใหม่, พลังเพื่อไทย, พลังไทยรักชาติ, ฟิวชัน, ภูมิใจไทย, รวมพลังประชาชน, รักชาติ, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อชีวิตใหม่, เพื่อบ้านเมือง, เศรษฐกิจ, เสรีรวมไทย, แผ่นดินธรรม, แรงงานสร้างชาติ, โอกาสใหม่, ไทยก้าวหน้า, ไทยก้าวใหม่, ไทยชนะ, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยภักดี, ไทยรวมไทย, ไทยสร้างไทย, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: ปงเตา | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: ปงเตา | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: ปงเตา | หน่วยที่: 12
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: หลวงใต้ | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: งาว | ตำบล: แม่ตีบ | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ท้องที่ไทย, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: ทุ่งฮั้ว | หน่วยที่: 10
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: ทุ่งฮั้ว | หน่วยที่: 11
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังซ้าย | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังทรายคำ | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังทรายคำ | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังทรายคำ | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังทรายคำ | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังทอง | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังแก้ว | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังแก้ว | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังใต้ | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังใต้ | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: วังเหนือ | ตำบล: วังใต้ | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: ทุ่งกว๋าว | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: ทุ่งกว๋าว | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: ทุ่งกว๋าว | หน่วยที่: 10
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: ทุ่งกว๋าว | หน่วยที่: 15
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 31 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 26 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชาติไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยทรัพย์ทวี, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 8
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: บ้านขอ | หน่วยที่: 13
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: หัวเมือง | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: หัวเมือง | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: หัวเมือง | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: หัวเมือง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 56 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 1 รายชื่อ: ท้องที่ไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: เมืองปาน | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: เมืองปาน | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: เมืองปาน | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 23 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 34 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ทางเลือกใหม่, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ประชาธิปไตยใหม่, ปวงชนไทย, พลวัต, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, มิติใหม่, รวมพลังประชาชน, รวมใจไทย, รวมไทยสร้างชาติ, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชาติไทย, เพื่อชีวิตใหม่, เพื่อไทย, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ใหม่, ไทยก้าวหน้า, ไทยชนะ, ไทยทรัพย์ทวี, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 24 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 33 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ทางเลือกใหม่, ประชาชน, ประชาธิปไตยใหม่, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลวัต, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อชาติไทย, เพื่อบ้านเมือง, เพื่อไทย, แผ่นดินธรรม, โอกาสใหม่, ใหม่, ไทยก้าวใหม่, ไทยทรัพย์ทวี, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองปาน | ตำบล: แจ้ซ้อน | หน่วยที่: 13
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
📌 [เขต] อำเภอ: เมืองลำปาง | ตำบล: บ้านเสด็จ | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 17 บรรทัด (จากที่ควรมี 7 บรรทัด)
   🚨 แปลกปลอม/ผิดสเปก 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านเสด็จ | หน่วยที่: 7
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านเสด็จ | หน่วยที่: 12
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านเสด็จ | หน่วยที่: 13
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านเสด็จ | หน่วยที่: 19
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านแลง | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 56 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 1 รายชื่อ: ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านแลง | หน่วยที่: 9
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: เมืองลำปาง | ตำบล: บ้านแลง | หน่วยที่: 12
   📊 มีข้อมูลทั้งหมด 56 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 1 รายชื่อ: ประชาธิปไตยใหม่
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: ทุ่งผึ้ง | หน่วยที่: 3
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: ทุ่งผึ้ง | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: บ้านสา | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: บ้านสา | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: ปงดอน | หน่วยที่: 5
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: เมืองมาย | หน่วยที่: 1
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ท้องที่ไทย, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: เมืองมาย | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: แจ้ห่ม(นอกเขต) | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: แจ้ห่ม(ในเขต) | หน่วยที่: 2
   📊 มีข้อมูลทั้งหมด 47 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 10 รายชื่อ: ทางเลือกใหม่, ประชาธิปไตยใหม่, พลวัต, มิติใหม่, รวมใจไทย, รวมไทยสร้างชาติ, เพื่อชาติไทย, เพื่อไทย, ใหม่, ไทยทรัพย์ทวี
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: แม่สุก | หน่วยที่: 4
   📊 มีข้อมูลทั้งหมด 33 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 24 รายชื่อ: ก้าวอิสระ, ครูไทยเพื่อประชาชน, คลองไทย, ท้องที่ไทย, ประชากรไทย, ประชาชาติ, ประชาธิปัตย์, ปวงชนไทย, พลังสังคมใหม่, พลังเพื่อไทย, ฟิวชัน, รวมพลังประชาชน, วิชชั่นใหม่, สร้างอนาคตไทย, สังคมประชาธิปไตย, อนาคตไทย, เพื่อชีวิตใหม่, เศรษฐกิจ, เสรีรวมไทย, แรงงานสร้างชาติ, ไทยก้าวหน้า, ไทยชนะ, ไทยภักดี, ไทรวมพลัง
------------------------------------------------------------
📌 [บช] อำเภอ: แจ้ห่ม | ตำบล: แม่สุก | หน่วยที่: 6
   📊 มีข้อมูลทั้งหมด 34 บรรทัด (จากที่ควรมี 57 บรรทัด)
   ❌ ขาดหายไป 23 รายชื่อ: กรีน, กล้าธรรม, ความหวังใหม่, ประชาชน, ประชาอาสาชาติ, ประชาไทย, พร้อม, พลังธรรมใหม่, พลังประชารัฐ, พลังไทยรักชาติ, ภูมิใจไทย, รักชาติ, เครือข่ายชาวนาแห่งประเทศไทย, เป็นธรรม, เพื่อบ้านเมือง, แผ่นดินธรรม, โอกาสใหม่, ไทยก้าวใหม่, ไทยธรรม, ไทยพร้อม, ไทยพิทักษ์ธรรม, ไทยรวมไทย, ไทยสร้างไทย
------------------------------------------------------------
จำนวนที่ขาดรวมคิดเป็น: 1092
"""

def parse_error_log(log_text):
    missing_dict = {}
    blocks = log_text.strip().split('------------------------------------------------------------')
    for block in blocks:
        if not block.strip(): continue
        meta = re.search(r'📌 \[(.*?)\] อำเภอ: (.*?) \| ตำบล: (.*?) \| หน่วยที่: (\d+)', block)
        missing = re.search(r'❌ ขาดหายไป \d+ รายชื่อ: (.*)', block)
        if meta and missing:
            t_val, d, sd, u = meta.groups()
            parties = [p.strip() for p in missing.group(1).split(',')]
            missing_dict[(t_val.strip(), d.strip(), sd.strip(), u.strip())] = set(parties)
    return missing_dict

# =========================================================
# 2. เตรียมข้อมูล Master List
# =========================================================
with open('election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

d = dict()
for p in data['provinces']:
    if p['name'] == 'ลำปาง':
        for a in p['areas']:
            for unit in a['stations']:
                if unit['district'] not in d:
                    d[unit['district']] = dict()
                d[unit['district']][unit['subdistrict']] = 1

# เพิ่มข้อมูลพิเศษ
d['นอกเขต'] = dict()
for i in range(1,14): d['นอกเขต'][f"ชุดที่ {i}"] = 1
d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)

master_districts = list(d.keys())
master_subdistricts = []
for v in d.values(): master_subdistricts += list(v.keys())

# =========================================================
# 3. ฟังก์ชันตัวช่วย
# =========================================================
def extract_file_info(filename):
    sub_district_match = re.search(r'(?:ตำบล|ต\.)\s*([\u0E00-\u0E7F]+)', filename)
    sub_district = sub_district_match.group(1) if sub_district_match else "-1"
    if sub_district == "-1":
        set_match = re.search(r'(ชุดที่\s*\d+)', filename)
        sub_district = set_match.group(1) if set_match else "-1"
    unit_match = re.search(r'หน่วยที่_?(\d+)', filename)
    unit_number = unit_match.group(1) if unit_match else "-1"
    return sub_district, unit_number

def get_fuzzy_match(text, valid_choices, threshold=60):
    if not text or text == "-1" or text == "ไม่ระบุ": return None
    match = process.extractOne(text, valid_choices, score_cutoff=threshold)
    return match[0] if match else None

# ฟังก์ชันดึงคะแนน (แบบผ่อนปรน ไม่บังคับคอลัมน์แรกต้องเป็นตัวเลข)
def extract_scores_relaxed(content, type_val):
    content = re.sub(r'(\d),(\d)', r'\1\2', content)
    trans = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    content = content.translate(trans)
    
    scores = []
    rows = re.findall(r'<tr.*?>(.*?)</tr>', content, re.IGNORECASE | re.DOTALL)
    for row in rows:
        cells = re.findall(r'<(?:td|th).*?>(.*?)</(?:td|th)>', row, re.IGNORECASE | re.DOTALL)
        cells = [re.sub(r'<.*?>', '', c).strip() for c in cells]
        
        if len(cells) >= 3:
            name = cells[1].strip()
            score_str = cells[2] if type_val == "บช" else (cells[3] if len(cells) >= 4 else "")
            score_match = re.search(r'\d+', score_str)
            score = score_match.group(0) if score_match else "-"
            if score == "0" or not score_str.strip(): score = "-"
            scores.append((name, score))
    return scores

def load_existing_records(csv_path):
    existing = set()
    if csv_path.exists():
        with open(csv_path, 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                try:
                    key = (row['type'].strip(), row['province'].strip(), 
                           row['district'].strip(), row['sub-district'].strip(),
                           str(row['unit_number']).strip(), row['name'].strip())
                    existing.add(key)
                except KeyError: pass
    return existing

# =========================================================
# 4. ฟังก์ชันหลัก (Incremental Update & Recovery)
# =========================================================
def main():
    source_dir = Path("ocr-result")
    output_dir = Path("ocr-result1")
    output_dir.mkdir(exist_ok=True)
    csv2_path = output_dir / "master_results1.csv"

    # 1. โหลด Error Log เพื่อหาเป้าหมาย
    missing_dict = parse_error_log(error_log)
    print(f"🎯 โหลดเป้าหมายจาก Log: พบหน่วยที่มีปัญหา {len(missing_dict)} หน่วย")

    # 2. โหลด CSV เดิมเข้าหน่วยความจำ (เพื่อเช็คว่าเคยเขียนหรือยัง)
    existing_records = load_existing_records(csv2_path)
    print(f"📥 โหลดข้อมูล CSV เดิม: พบ {len(existing_records)} เรคคอร์ด\n")

    files = sorted(list(source_dir.glob("**/*.txt")))
    
    with open(csv2_path, "a", encoding="utf-8") as fw2:
        for file in files:
            structure = file.parent.parts
            raw_district = structure[1] if len(structure) > 1 else "ไม่ระบุ"
            raw_sub_district, unit_num = extract_file_info(file.stem)
            if raw_sub_district == "-1" and len(structure) > 2:
                raw_sub_district = structure[2]

            type_val = "บช" if "บช" in file.name else "เขต"

            # 3. จัดการปรับชื่อ District / Sub-district ให้ถูกต้อง
            district = raw_district if raw_district in master_districts else (get_fuzzy_match(raw_district, master_districts, 70) or raw_district)
            sub_district = raw_sub_district if raw_sub_district in master_subdistricts else (get_fuzzy_match(raw_sub_district, master_subdistricts, 70) or raw_sub_district)

            unit_key = (type_val, district, sub_district, str(unit_num))

            # เช็คว่าหน่วยนี้ มีรายชื่อหายไปตาม Log หรือไม่?
            if unit_key in missing_dict:
                missing_parties = missing_dict[unit_key]
                if not missing_parties: continue # กู้ครบแล้ว ข้ามเลย

                try:
                    with open(file, 'r', encoding="utf-8") as f:
                        content = f.read()

                    scores_relaxed = extract_scores_relaxed(content, type_val)
                    recovered_count = 0

                    for ocr_name, score in scores_relaxed:
                        # สร้าง Key ของบรรทัดนี้ เพื่อเช็คว่า "เคยบันทึกไปหรือยัง"
                        check_key = (type_val, "ลำปาง", district, sub_district, str(unit_num), ocr_name)
                        
                        # 4.1 ถ้าเคยเซฟแล้ว -> ข้ามทันที (ไม่ต้องอัปเดตซ้ำ)
                        if check_key in existing_records:
                            continue

                        # 4.2 ถ้ายังไม่เคยเซฟ -> มองหาว่าพรรคนี้อยู่ใน Log ที่หายไปไหม?
                        best_match = None
                        if ocr_name in missing_parties:
                            best_match = ocr_name # ตรงเป๊ะ!
                        else:
                            # 4.3 ถ้าหาไม่เจอตรงๆ -> ค่อยใช้ Fuzzy Match เทียบเฉพาะเป้าหมาย
                            best_match = get_fuzzy_match(ocr_name, list(missing_parties), threshold=60)

                        # ถ้าหาเจอเป้าหมาย (ทั้งแบบตรงเป๊ะ และ แบบ Fuzzy) ให้บันทึก
                        if best_match:
                            correct_key = (type_val, "ลำปาง", district, sub_district, str(unit_num), best_match)
                            
                            if correct_key not in existing_records:
                                # เซฟข้อมูลที่แก้ไขชื่อ Sub-district, District และ พรรค ถูกต้องแล้ว
                                row = f"{type_val},ลำปาง,{district},{sub_district},{unit_num},{best_match},{score}\n"
                                fw2.write(row)
                                
                                existing_records.add(correct_key)
                                missing_parties.remove(best_match) # ติ๊กออกจาก Log ว่าเจอแล้ว
                                recovered_count += 1
                                
                                if best_match != ocr_name:
                                    print(f"   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: '{ocr_name}' -> '{best_match}'")

                    if recovered_count > 0:
                        print(f"✅ อัปเดต {district} -> {sub_district} (หน่วย {unit_num}): กู้คืนได้ {recovered_count} พรรค")

                except Exception as e:
                    print(f"❌ Error on {file.stem}: {e}")

if __name__ == "__main__":
    main()
    print("-" * 50)
    print("✨ อัปเดตข้อมูลส่วนที่ขาดหายเข้าสู่ master_results1.csv เรียบร้อยแล้ว!")

🎯 โหลดเป้าหมายจาก Log: พบหน่วยที่มีปัญหา 71 หน่วย
📥 โหลดข้อมูล CSV เดิม: พบ 21648 เรคคอร์ด

   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: 'ประชาธิปโดยใหม่' -> 'ประชาธิปไตยใหม่'
✅ อัปเดต งาว -> นาแก (หน่วย 4): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> บ้านร้อง (หน่วย 1): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> บ้านร้อง (หน่วย 9): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> บ้านอ้อน (หน่วย 1): กู้คืนได้ 10 พรรค
   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: 'ประชาธิปโดยใหม่' -> 'ประชาธิปไตยใหม่'
✅ อัปเดต งาว -> บ้านอ้อน (หน่วย 2): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> บ้านอ้อน (หน่วย 3): กู้คืนได้ 2 พรรค
   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: 'ประชาธิปโดยใหม่' -> 'ประชาธิปไตยใหม่'
✅ อัปเดต งาว -> บ้านอ้อน (หน่วย 5): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> บ้านแหง (หน่วย 7): กู้คืนได้ 10 พรรค
✅ อัปเดต งาว -> หลวงใต้ (หน่วย 3): กู้คืนได้ 10 พรรค
   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: 'ท้องที่ไทยใหม่' -> 'ท้องที่ไทย'
✅ อัปเดต งาว -> แม่ตีบ (หน่วย 7): กู้คืนได้ 10 พรรค
   🪄 ซ่อมชื่อพรรคด้วย Fuzzy: 'ประชาธิปโดยใหม่' -> 'ประชาธิปไตยใหม่'
✅ อัปเดต งาว -> ปงเตา (หน่วย 1): กู้คืนได้ 